# 🌉 SHMS AI Anomaly Detection — Finger Bridge
## Phase 3B: Isolation Forest

**Perbedaan dari Phase 3A (LSTM)**:

| Aspek | LSTM Autoencoder | Isolation Forest |
|---|---|---|
| Input | Raw time-series (1000×17) | Fitur statistik (136 fitur) |
| Deteksi | Anomali **temporal** | Anomali **statistik** |
| Hardware | GPU direkomendasikan | CPU sudah cukup |
| Training | ~20 menit | **< 5 menit** |
| Kelebihan | Pola waktu kompleks | Ringan, interpretable |

> Keduanya digabung di Phase 4 (Ensemble Fusion)

---

## 0. Setup & Cek Environment

In [ ]:
# Isolation Forest bisa jalan di CPU — tidak perlu GPU
import platform, psutil
import os

cpu_count = os.cpu_count()
ram_gb    = psutil.virtual_memory().total / 1e9
print(f'CPU cores : {cpu_count}')
print(f'RAM       : {ram_gb:.1f} GB')
print(f'OS        : {platform.system()}')
print(f'\n✅ Isolation Forest bisa jalan optimal di environment ini')


---
## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# ══════════════════════════════════════════════════════
# SESUAIKAN PATH INI
# ══════════════════════════════════════════════════════
GDRIVE_PROJECT = Path('/content/drive/MyDrive/shms-ai-anomaly-detection-fingerbridge')

CODE_DIR      = GDRIVE_PROJECT / '03_code'
DATA_PROC_DIR = GDRIVE_PROJECT / '02_data' / 'processed'
MODEL_DIR     = GDRIVE_PROJECT / '04_models'
RESULTS_DIR   = GDRIVE_PROJECT / '05_results'

for d in [MODEL_DIR, RESULTS_DIR, RESULTS_DIR/'figures']:
    d.mkdir(parents=True, exist_ok=True)

print('📁 Folder proyek:')
for label, path in [('Code','CODE_DIR'),('Data','DATA_PROC_DIR'),
                     ('Models','MODEL_DIR'),('Results','RESULTS_DIR')]:
    p = eval(path)
    print(f'  {label:10s}: {p}  {"✅" if p.exists() else "❌"}')

npy_files = sorted(DATA_PROC_DIR.glob('*_X.npy')) if DATA_PROC_DIR.exists() else []
print(f'\n📊 Data processed: {len(npy_files)} hari tersedia')


---
## 2. Import Library

In [ ]:
import sys, json, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import seaborn as sns
from scipy import stats

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
import joblib

print('✅ Import selesai')


---
## 3. Konfigurasi

In [ ]:
MAIN_CHANNELS = [
    'FB_AC_PY1T_01_BX', 'FB_AC_PY1T_01_BY',
    'FB_AC_PY1D_01_BX', 'FB_AC_PY1D_01_BY',
    'FB_AC_S2M_01_BZ',
    'FB_AC_S1Q1_01_BX', 'FB_AC_S1Q1_01_BZ',
    'FB_AC_S3Q3_01_BY', 'FB_AC_S3Q3_01_BZ',
    'FB_CA_L17_EZ', 'FB_CA_R17_EZ',
    'FB_CA_L22_EZ', 'FB_CA_R22_EZ',
    'FB_CA_L46_EZ', 'FB_CA_R46_EZ',
    'FB_CA_L02_EZ', 'FB_CA_L55_EZ',
]
CHANNEL_ALIAS = {
    'FB_AC_PY1T_01_BX':'AC_PY1T_BX','FB_AC_PY1T_01_BY':'AC_PY1T_BY',
    'FB_AC_PY1D_01_BX':'AC_PY1D_BX','FB_AC_PY1D_01_BY':'AC_PY1D_BY',
    'FB_AC_S2M_01_BZ' :'AC_S2M_BZ',
    'FB_AC_S1Q1_01_BX':'AC_S1Q1_BX','FB_AC_S1Q1_01_BZ':'AC_S1Q1_BZ',
    'FB_AC_S3Q3_01_BY':'AC_S3Q3_BY','FB_AC_S3Q3_01_BZ':'AC_S3Q3_BZ',
    'FB_CA_L17_EZ':'CA_L17','FB_CA_R17_EZ':'CA_R17',
    'FB_CA_L22_EZ':'CA_L22','FB_CA_R22_EZ':'CA_R22',
    'FB_CA_L46_EZ':'CA_L46','FB_CA_R46_EZ':'CA_R46',
    'FB_CA_L02_EZ':'CA_L02','FB_CA_L55_EZ':'CA_L55',
}
N_CHANNELS  = len(MAIN_CHANNELS)
WINDOW_SIZE = 1000

HP = {
    'n_estimators' : 200,
    'max_samples'  : 0.8,
    'contamination': 0.05,
    'max_features' : 1.0,
    'random_state' : 42,
    'n_jobs'       : -1,
    'threshold_pct': 95,
}

FEATURE_NAMES = ['mean','std','min','max','ptp','rms','skewness','kurtosis']
print(f'Channels: {N_CHANNELS} | Fitur/channel: {len(FEATURE_NAMES)}')
print(f'Total fitur: {N_CHANNELS * len(FEATURE_NAMES)}')


---
## 4. Load Data

In [ ]:
def load_split(split, data_dir, normal_only=False, max_windows=None):
    summary_path = data_dir / 'processing_summary.csv'
    if not summary_path.exists():
        print(f'  ⚠️  [{split}] Mode simulasi')
        n  = {'train':5000,'val':1000,'test':1000}[split]
        X  = np.random.randn(n, WINDOW_SIZE, N_CHANNELS).astype('float32')
        y  = np.zeros(n, dtype='int8')
        if split != 'train':
            abn = np.random.choice(n, n//10, replace=False)
            X[abn] += (np.random.randn(len(abn),WINDOW_SIZE,N_CHANNELS)*3).astype('float32')
            y[abn] = 1
        return X, y
    summary = pd.read_csv(summary_path)
    days = summary[(summary['split']==split)&(summary['status']=='ok')]['date'].tolist()
    X_list, y_list = [], []
    for d in sorted(days):
        xp, yp = data_dir/f'{d}_X.npy', data_dir/f'{d}_y.npy'
        if xp.exists():
            X_list.append(np.load(xp)); y_list.append(np.load(yp))
    X = np.concatenate(X_list).astype('float32')
    y = np.concatenate(y_list)
    if normal_only: X,y = X[y==0], y[y==0]
    if max_windows and len(X)>max_windows:
        idx = np.sort(np.random.choice(len(X),max_windows,replace=False))
        X,y = X[idx],y[idx]
    return X, y

print('📂 Loading data...')
X_train, y_train = load_split('train', DATA_PROC_DIR, normal_only=True)
X_val,   y_val   = load_split('val',   DATA_PROC_DIR)
X_test,  y_test  = load_split('test',  DATA_PROC_DIR)

print(f'\n  Train  : {X_train.shape}  — hanya normal')
print(f'  Val    : {X_val.shape}  — {y_val.sum()} abnormal')
print(f'  Test   : {X_test.shape}  — {y_test.sum()} abnormal')


---
## 5. Feature Extraction

Setiap window (1000 × 17) → 136 fitur statistik:

| Fitur | Keterangan |
|---|---|
| mean | Rata-rata nilai |
| std | Standar deviasi |
| min / max | Nilai ekstrem |
| ptp | Peak-to-peak (max−min) |
| rms | Root mean square (energi sinyal) |
| skewness | Kemiringan distribusi |
| kurtosis | Ketajaman distribusi (outlier sensitivity) |


In [ ]:
def extract_features(X, channel_names):
    """Ekstrak 8 fitur statistik per channel per window."""
    n_windows, win_size, n_ch = X.shape
    records = []
    for i in range(n_windows):
        row = {}
        for j, ch in enumerate(channel_names):
            alias   = CHANNEL_ALIAS.get(ch, ch.replace('FB_',''))
            s       = X[i,:,j]
            s_clean = s[~np.isnan(s)]
            if len(s_clean) < 2:
                for f in FEATURE_NAMES: row[f'{alias}_{f}'] = 0.0
                continue
            row[f'{alias}_mean']     = float(s_clean.mean())
            row[f'{alias}_std']      = float(s_clean.std())
            row[f'{alias}_min']      = float(s_clean.min())
            row[f'{alias}_max']      = float(s_clean.max())
            row[f'{alias}_ptp']      = float(s_clean.max()-s_clean.min())
            row[f'{alias}_rms']      = float(np.sqrt((s_clean**2).mean()))
            row[f'{alias}_skewness'] = float(stats.skew(s_clean))
            row[f'{alias}_kurtosis'] = float(stats.kurtosis(s_clean))
        records.append(row)
    return pd.DataFrame(records)

print('⚙️  Mengekstrak fitur...')
t0 = time.time()
feat_train = extract_features(X_train, MAIN_CHANNELS)
feat_val   = extract_features(X_val,   MAIN_CHANNELS)
feat_test  = extract_features(X_test,  MAIN_CHANNELS)
elapsed = time.time()-t0

print(f'✅ Feature extraction selesai ({elapsed:.1f}s)')
print(f'  Shape: {feat_train.shape[0]:,} windows × {feat_train.shape[1]} fitur')
print(f'\nContoh fitur:')
feat_train.head(2)


---
## 6. Feature Scaling

In [ ]:
scaler     = StandardScaler()
ft_scaled  = scaler.fit_transform(feat_train.fillna(0))
fv_scaled  = scaler.transform(feat_val.fillna(0))
ftest_scaled = scaler.transform(feat_test.fillna(0))

print(f'✅ Scaling selesai')
print(f'  Train mean (post-scale): {ft_scaled.mean():.4f} (diharapkan ~0)')
print(f'  Train std  (post-scale): {ft_scaled.std():.4f}  (diharapkan ~1)')


---
## 7. Training Isolation Forest
> Estimasi waktu: **< 3 menit** di Colab

In [ ]:
print('🌲 Training Isolation Forest...')
print(f'  n_estimators = {HP["n_estimators"]} pohon')
print(f'  max_samples  = {HP["max_samples"]} ({int(HP["max_samples"]*len(feat_train)):,} sampel per pohon)')
print(f'  contamination= {HP["contamination"]}')

t0 = time.time()
iforest = IsolationForest(
    n_estimators  = HP['n_estimators'],
    max_samples   = HP['max_samples'],
    contamination = HP['contamination'],
    max_features  = HP['max_features'],
    random_state  = HP['random_state'],
    n_jobs        = HP['n_jobs'],
)
iforest.fit(ft_scaled)
elapsed = time.time()-t0

print(f'\n✅ Training selesai dalam {elapsed:.1f} detik')
print(f'  Pohon dibuat: {len(iforest.estimators_)}')


---
## 8. Kalibrasi Threshold

`score_samples()` mengembalikan nilai negatif — semakin negatif = semakin anomali.
Kita balikkan tandanya agar lebih intuitif: **semakin tinggi = semakin anomali**.


In [ ]:
# Hitung anomaly score pada validation set
# score_samples: lebih rendah = lebih anomali → balik tanda
scores_val = -iforest.score_samples(fv_scaled)

sc_normal = scores_val[y_val == 0]
threshold = np.percentile(sc_normal, HP['threshold_pct'])

print(f'Statistik score (data normal):')
for p in [50, 75, 90, 95, 99]:
    marker = ' ← threshold' if p == HP['threshold_pct'] else ''
    print(f'  P{p:2d} = {np.percentile(sc_normal,p):.5f}{marker}')
print(f'\n✅ Threshold: {threshold:.5f} (P{HP["threshold_pct"]})')


In [ ]:
# Plot distribusi score
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sc_abnorm = scores_val[y_val==1] if y_val.sum()>0 else np.array([])
axes[0].hist(sc_normal, bins=60, alpha=0.75, color='#378ADD', label='Normal', density=True)
if len(sc_abnorm):
    axes[0].hist(sc_abnorm, bins=40, alpha=0.75, color='#E24B4A', label='Abnormal', density=True)
axes[0].axvline(threshold, color='#BA7517', lw=2.5, linestyle='--',
                label=f'Threshold P{HP["threshold_pct"]}={threshold:.4f}')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribusi Score — Validation Set')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

pcts = np.arange(50, 100, 0.5)
axes[1].plot(pcts, [np.percentile(sc_normal,p) for p in pcts],
             color='#378ADD', lw=2)
axes[1].axvline(HP['threshold_pct'], color='#BA7517', lw=1.5,
                linestyle='--', label=f'P{HP["threshold_pct"]}')
axes[1].set_xlabel('Persentil'); axes[1].set_ylabel('Score value')
axes[1].set_title('Kurva Persentil Score Normal')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Isolation Forest — Threshold Calibration', fontsize=12)
plt.tight_layout()
p = RESULTS_DIR/'figures'/'iforest_threshold.png'
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print(f'Disimpan: {p}')


---
## 9. Evaluasi pada Data Test

In [ ]:
scores_test = -iforest.score_samples(ftest_scaled)
s_min, s_max = scores_test.min(), scores_test.max()
scores_norm  = (scores_test-s_min)/(s_max-s_min+1e-9)
y_pred       = (scores_test > threshold).astype(int)

precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
auc       = roc_auc_score(y_test, scores_norm) if y_test.sum()>0 else 0.0
cm        = confusion_matrix(y_test, y_pred)

print('='*50)
print('  HASIL EVALUASI — DATA TEST')
print('='*50)
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1-score   : {f1:.4f}  ← metrik utama')
print(f'  AUC-ROC    : {auc:.4f}')
print(f'\n  Confusion Matrix:')
print(f'    TN={cm[0,0]:6,}  FP={cm[0,1]:6,}')
print(f'    FN={cm[1,0]:6,}  TP={cm[1,1]:6,}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Normal','Abnormal'])
axes[0].set_yticklabels(['Normal','Abnormal'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[0].text(j,i,f'{cm[i,j]:,}',ha='center',va='center',
                     fontsize=14,fontweight='bold',
                     color='white' if cm[i,j]>cm.max()//2 else 'black')
plt.colorbar(im, ax=axes[0])

if y_test.sum()>0:
    fpr,tpr,_ = roc_curve(y_test, scores_norm)
    axes[1].plot(fpr,tpr,color='#1D9E75',lw=2,label=f'AUC={auc:.4f}')
    axes[1].plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title('ROC Curve')
    axes[1].legend(); axes[1].grid(True,alpha=0.3)

plt.suptitle('Isolation Forest — Evaluasi Data Test', fontsize=12)
plt.tight_layout()
p = RESULTS_DIR/'figures'/'iforest_evaluation.png'
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.show()


---
## 10. Feature Importance

Fitur mana yang paling "informatif" untuk mendeteksi anomali?
Ini berguna untuk bagian **Discussion** di paper.


In [ ]:
# Estimasi importance: fitur yang sering dipakai di level pohon dangkal
importances = np.zeros(feat_train.shape[1])
feat_names  = feat_train.columns.tolist()

for tree in iforest.estimators_:
    stack = [(0,0)]
    depth_arr = np.zeros(tree.tree_.feature.shape[0])
    while stack:
        node, d = stack.pop()
        depth_arr[node] = d
        if tree.tree_.children_left[node] != -1:
            stack.append((tree.tree_.children_left[node],  d+1))
            stack.append((tree.tree_.children_right[node], d+1))
    for node in range(tree.tree_.node_count):
        f = tree.tree_.feature[node]
        if 0 <= f < len(feat_names):
            importances[f] += 1.0 / (depth_arr[node] + 1)

importances /= importances.sum()
top_n   = 20
top_idx = np.argsort(importances)[-top_n:][::-1]

colors = ['#378ADD' if n.startswith('AC') else '#1D9E75'
          if n.startswith('CA') else '#888780'
          for n in [feat_names[i] for i in top_idx]]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_n), importances[top_idx[::-1]], color=colors[::-1])
ax.set_yticks(range(top_n))
ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=8)
ax.set_xlabel('Relative Importance')
ax.set_title(f'Top {top_n} Features — Isolation Forest\n'
             '(Biru=Accelerometer, Hijau=Cable)')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
p = RESULTS_DIR/'figures'/'iforest_feature_importance.png'
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print(f'Disimpan: {p}')

# Tampilkan top 10 sebagai tabel
top10_df = pd.DataFrame({
    'feature'   : [feat_names[i] for i in top_idx[:10]],
    'importance': importances[top_idx[:10]].round(5)
})
print(top10_df.to_string(index=False))


---
## 11. Simpan Model ke GDrive

In [ ]:
# Simpan model dan scaler
joblib.dump(iforest, MODEL_DIR/'isolation_forest.pkl')
joblib.dump(scaler,  MODEL_DIR/'iforest_scaler.pkl')

# Simpan feature names
with open(MODEL_DIR/'iforest_feature_names.json','w') as f:
    json.dump(feat_train.columns.tolist(), f)

# Simpan threshold
with open(MODEL_DIR/'iforest_threshold.json','w') as f:
    json.dump({
        'threshold'        : float(threshold),
        'threshold_pct'    : HP['threshold_pct'],
        'score_mean_normal': float(sc_normal.mean()),
        'score_std_normal' : float(sc_normal.std()),
    }, f, indent=2)

# Simpan metrics
pd.DataFrame([{
    'model':'Isolation_Forest','threshold':threshold,
    'precision':precision,'recall':recall,'f1':f1,'auc':auc,
    'tn':cm[0,0],'fp':cm[0,1],'fn':cm[1,0],'tp':cm[1,1],
    'n_features':feat_train.shape[1],
    'n_estimators':HP['n_estimators'],
}]).to_csv(RESULTS_DIR/'iforest_metrics.csv', index=False)

# Update experiment log
log_path = GDRIVE_PROJECT/'05_results'/'experiment_log.csv'
log_row  = pd.DataFrame([{
    'run_id'    : pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'),
    'timestamp' : pd.Timestamp.now().isoformat(),
    'model'     : 'Isolation_Forest',
    'window_size': 1000,
    'threshold' : round(float(threshold),6),
    'precision' : round(precision,4),
    'recall'    : round(recall,4),
    'f1'        : round(f1,4),
    'auc'       : round(auc,4),
    'notes'     : f'n_est={HP["n_estimators"]},features={feat_train.shape[1]}',
}])
if log_path.exists():
    log_row.to_csv(log_path, mode='a', header=False, index=False)
else:
    log_row.to_csv(log_path, index=False)

print('✅ Semua file tersimpan ke GDrive:')
for fname in ['isolation_forest.pkl','iforest_scaler.pkl',
               'iforest_threshold.json','iforest_feature_names.json']:
    p = MODEL_DIR/fname
    if p.exists():
        print(f'  {fname:40s} {p.stat().st_size/1024:6.1f} KB')


---
## 12. Perbandingan LSTM vs Isolation Forest

In [ ]:
# Load hasil LSTM jika ada
lstm_path = RESULTS_DIR / 'lstm_metrics.csv'
if_path   = RESULTS_DIR / 'iforest_metrics.csv'

rows = []
for path in [lstm_path, if_path]:
    if path.exists():
        rows.append(pd.read_csv(path).iloc[0])

if rows:
    compare_df = pd.DataFrame(rows)[['model','precision','recall','f1','auc']]
    print('\n  Perbandingan model:')
    print(compare_df.to_string(index=False))
    print('\n  (Phase 4 akan menggabungkan keduanya → Ensemble Fusion)')
else:
    print('  Jalankan Phase 3A dulu untuk melihat perbandingan')

print('\n' + '='*50)
print('  RINGKASAN — ISOLATION FOREST')
print('='*50)
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1-score   : {f1:.4f}')
print(f'  AUC-ROC    : {auc:.4f}')
print(f'  n_features : {feat_train.shape[1]}')
print(f'  Threshold  : {threshold:.5f} (P{HP["threshold_pct"]})')
print('='*50)
print('\n  Next: Phase 3C — GNN')
print('        notebook: SHMS_Phase3C_GNN.ipynb')
